In [ ]:
import os, mne
import numpy as np
import matplotlib.pyplot as plt
from preprocFuncs import importData
from flagger import badChanFixer, icaFixer

%matplotlib qt
%load_ext autoreload
%autoreload 2

In [ ]:
# Define the path to the raw data
bidsRoot = os.path.join(os.sep, 'd', 'DATD', 'datd', 'MEG_MGS', 'MEG_BIDS')
derivRoot = os.path.join(os.sep, 'd', 'DATD', 'datd', 'MEG_MGS', 'MEG_BIDS', 'derivatives')
taskName = 'mgs'
subDirs = os.listdir(bidsRoot)
subDirs = [subDir for subDir in subDirs if subDir.startswith('sub-')]

sidx = 12
run = 8
fName_root = f'sub-{subDirs[sidx-1][-2:]}_task-{taskName}_'

# Load the data
raw, events, metadata = importData(bidsRoot, subDirs[sidx-1], run)

# Fix bad channels
raw = badChanFixer(raw, sidx, run)

In [ ]:
(events[events[:, 2] == 6, 0] - events[events[:, 2] == 5, 0]) / 500

In [ ]:
metadata.head(30)

In [ ]:
raw.plot()

In [ ]:
raw.compute_psd(fmax=100).plot()

In [ ]:
# Perform ICA
raw_ica = raw.copy()
raw_ica.load_data().filter(l_freq=1, h_freq=55, fir_design='firwin')
ica = mne.preprocessing.ICA(n_components=0.95, 
                            random_state=42,
                              max_iter=800).fit(raw_ica, 
                                                reject=dict(mag=4e-12))

In [ ]:
longMetadata.head()

In [ ]:
ica.plot_components()
ica.plot_sources(raw_ica)

In [ ]:
raw_clean = icaFixer(raw, ica, sidx, run)

In [ ]:
fig =  mne.viz.plot_alignment(
    raw_clean.info,
    dig=True,
    eeg=False,
    # meg=True,
    surfaces=[],
    meg=['helmet', 'sensors'],
    coord_frame='meg',
)

In [ ]:
# Epoch the data
longIdx = metadata.query('islong == True').index
longEvents = events[longIdx]
longMetadata = metadata.loc[longIdx]
epochsLong = mne.Epochs(
    raw_clean,
    longEvents,
    event_id=2,
    tmin=-1.5,
    tmax=4,
    baseline=(-1, 0),
    preload=True,
    detrend=1,
    reject=dict(mag=4e-12),
    metadata=longMetadata,
    verbose=True,
)

In [ ]:
# Plot ERP
epochsLong_temp = epochsLong.copy()
# epochsLong_temp.filter(l_freq=1, h_freq=55, fir_design='firwin')
evokedLong = epochsLong_temp.average()
evokedLong.plot()

In [ ]:
# Plot 

In [ ]:
raw.plot_sensors(kind='3d', ch_type='mag')

In [ ]:
# Perform the ICA
raw_ica = raw.copy()
raw_ica.filter(l_freq=1, h_freq=None, n_jobs=-1)
ica = mne.preprocessing.ICA(n_components=0.95, random_state=42, max_iter=800).fit(raw_ica, reject=dict(mag=4e-12))

In [ ]:
raw_ica.plot_psd(fmax=100)

In [ ]:
ica.plot_components()

In [ ]:
ica.plot_sources(raw_ica)

In [ ]:
ica.plot_overlay(raw_ica)

In [ ]:
raw_clean = ica.apply(raw_ica)

In [ ]:
trigger_data = raw.get_data(picks=trigger_ch_names).astype(int)

# Combine channels into a single stim channel using binary encoding
combined_trigger = np.sum(trigger_data * (2 ** np.arange(trigger_data.shape[0])[:, None]), axis=0)
print(combined_trigger)

In [ ]:
plt.figure()
plt.plot(trigger_data.T)
plt.show()

In [ ]:
trigger_ch_names = [f"Trigger{ch+1}" for ch in range(6)]
events = []
for ch in trigger_ch_names:
    events_this = mne.find_events(raw, stim_channel=ch, shortest_event=1)
    eventid = ch[-1]
    events_this[:, 2] = eventid
    print(len(events_this))
    events.append(events_this)
# events = mne.find_events(raw, stim_channel=trigger_ch_names, shortest_event=1)
print(events)

In [ ]:
events[0]

In [ ]:
# Visualize events
mne.viz.plot_events(
    events,
    sfreq=raw.info['sfreq'],
)

In [ ]:
montage = mne.channels.make_standard_montage("kit160")
info.set_montage(montage)

In [ ]:


# Example gradData: Replace with your actual sensor positions
gradData = np.random.rand(160, 3) * 1e-3  # Positions in meters

# Create channel names
n_channels = gradData.shape[0]
ch_names = [f"MEG{ch + 1:03d}" for ch in range(n_channels)]

# Sampling frequency
sfreq = 1000  # Replace with your actual sampling frequency

# Channel types
ch_types = ["grad"] * n_channels

# Create the info object
info = mne.create_info(ch_names=ch_names, sfreq=sfreq, ch_types=ch_types)

# Assign sensor positions
for idx, ch in enumerate(info["chs"]):
    loc = np.zeros(12)  # Initialize with zeros
    loc[:3] = gradData[idx]  # Assign the 3D position from gradData
    ch["loc"] = loc

fig = mne.viz.plot_alignment(
    info,
    meg=["sensors", "helmet"],  # Show both sensors and helmet
    coord_frame="meg",          # Coordinate frame for KIT
    show_axes=True,             # Optional: Show coordinate axes
)
mne.viz.set_3d_view(fig, azimuth=50, elevation=90, distance=0.5)


In [ ]:
fig = mne.viz.plot_alignment(
    info,
    dig=False,
    eeg=False,
    surfaces=[],
    meg=["helmet", "sensors"],
    coord_frame="head",
)
mne.viz.set_3d_view(fig, azimuth=50, elevation=90, distance=0.5)

In [ ]:
gradData  = data['grad'][0][0]['chanpos'][0][0].T
print(gradData.shape)
# gradData = np.squeeze(gradData)

fig = plt.figure()
ax = fig.add_subplot(111, projection='3d')
ax.scatter(gradData[0, :], gradData[1, :], gradData[2, :])
plt.show()

In [ ]:
coilOri = data['grad'][0][0]['coilori'][0][0]
print(len(coilOri))

In [ ]:
gradData = data['grad'][0][0]['chanpos'][0][0].T
gradData = np.array(gradData)

# Create the montage
# Create a dictionary of channel names and positions
dict_ch = {}
for i in range(len(ch_names)):
    dict_ch[ch_names[i]] = gradData[:, i]

montage = mne.channels.make_dig_montage(ch_pos=dict_ch, coord_frame='head')

# Visualize the montage in 3D
info.set_channel_types({ch_name: 'grad' for ch_name in ch_names})
info.set_montage(montage, verbose=True)
# fig = mne.viz.plot_alignment(info)

In [ ]:
info['chs'][0]

In [ ]:
# Visualize montage
fig = plt.figure()
montage.plot(kind='3d')
plt.show()

In [ ]:
info.get_channel_types

In [ ]:
for i in range(len(ch_names)):
    info['chs'][i]['loc'] = gradData[i, :]
info['chs'][0]

In [ ]:
info.get_montage()

In [ ]:
info.get_montage()

In [ ]:
epoch_fpath = os.path.join(bidsRoot, subDirs[sidx], 'meg', f'{fName_root}short_epoched.mat')
epochs = read_epochs_fieldtrip(epoch_fpath, info=info, data_name="epocShort", trialinfo_column=0)

In [ ]:
epochedData = epochs.get_data()
bad_trials = np.any(np.isnan(epochedData), axis=(1, 2))
epochs.drop(bad_trials)

In [ ]:
epochs.event_id

In [ ]:
# Select trials with stimulus on left side
leftEventIds = ['130', '155', '180', '205', '230']
epochsLeft = epochs[epochs.events[:, 2] == int(leftEventIds[0])]
rightEventIds = ['0', '25', '50', '310', '335']
epochsRight = epochs[epochs.events[:, 2] == int(rightEventIds[0])]

In [ ]:
# Perform time-frequency analysis to extract power in alpha band
freqs = np.arange(8, 13, 1)  # Frequencies from 8 to 12 Hz

# Number of cycles (can be a single value or an array matching `freqs`)
n_cycles = freqs / 2  # For example, 2 cycles per frequency

# Perform time-frequency decomposition
power = mne.time_frequency.tfr_multitaper(
    epochs, 
    freqs=freqs, 
    n_cycles=n_cycles, 
    time_bandwidth=2.0,  # Time-frequency resolution trade-off
    return_itc=False,  # Don't return inter-trial coherence
    average=False      # Keep single-trial results
)

In [ ]:
epochs.get_data().shape

In [ ]:
from sklearn.svm import SVC
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split

power_data = np.mean(power.data, axis=2)
print(power_data.shape)

X = power_data  # Shape: (82 trials, 157 channels, 2850 time points)
# Downsample the data to 
X = X[:, :, ::10]  # Shape: (82 trials, 157 channels, 285 time points)
timeArray = power.times[::10]
# y = trial_labels  # Shape: (82,) (Trial labels)
y = epochs.events[:, 2]
y = [0 if str(yy) in rightEventIds else 1 for yy in y]

# List to store accuracies for each time point
# accuracies = np.empty((X.shape[2], X.shape[2]))
accuracies_train = []
accuracies_test = []

# Loop over each time point
for t in range(X.shape[2]):
    # Extract the data for the current time point
    X_t = X[:, :, t]  # Shape: (82 trials, 157 channels)
    # Divide the data into training and testing sets
    X_train, X_test, y_train, y_test = train_test_split(X_t, y, test_size=0.2, random_state=42)
    print(X_train.shape, X_test.shape, len(y_train), len(y_test))
    
    # Train the classifier
    clf = make_pipeline(SVC(kernel='linear'))
    clf.fit(X_train, y_train)

    # Test the classifier
    accuracies_train.append(clf.score(X_train, y_train))
    accuracies_test.append(clf.score(X_test, y_test))


In [ ]:
# Plot the accuracies
plt.figure()
plt.plot(timeArray, accuracies_train, label='Train')
plt.plot(timeArray, accuracies_test, label='Test')
plt.xlabel('Time (s)')
plt.ylabel('Accuracy')
plt.legend()
plt.show()

In [ ]:
# Get the power data
power_data = powerLeft.data
plt.figure()
plt.imshow(np.mean(power_data, axis=1), aspect='auto', origin='lower', cmap='RdBu_r')
plt.colorbar()
plt.show()

In [ ]:
np.min(power_data), np.max(power_data)

In [ ]:
# Visualize the power as a topomap
power_data = np.mean(power_data, axis=2)

# Create a montage


In [ ]:
# Extract channel positions and labels
ch_names = [label[0] for label in lay['label'][0]]  # Adjust based on structure
pos = lay['pos'][:, :2][0][0] # Adjust based on structure

pos_3d = np.hstack([pos, np.zeros((pos.shape[0], 1))])

# Create a montage
montage = make_dig_montage(ch_pos=dict(zip(ch_names, pos_3d)))

In [ ]:
data['grad'][0][0]

In [ ]:
rawM.plot(
    n_channels=10,
    scalings=dict(mag=1e-6),
)
plt.show()

In [ ]:
runIdx = 0
raw = mne.io.read_raw_fif(
    os.path.join(bidsRoot, subDirs[sidx], 'meg', recFiles[runIdx]),
    preload=True
)

In [ ]:
raw.ch_names

In [ ]:
# Read events from 'STI014' channel
events = mne.find_events(raw, stim_channel='STI 014')
print(events)

In [ ]:
raw.plot(
    n_channels=10,
    scalings=dict(mag=1e-6),
)
plt.show()

In [ ]:
data = raw.get_data()
print(data.shape)

In [ ]:
plt.figure()
plt.plot(data[-1, :].T)
plt.show()